# PhySim06 — Language-Guided Physical AI Agent

### Lab Description

This capstone combines the previous Genesis labs into one guarded physical AI loop:

`natural language → validated JSON plan → visual confirmation → IK/action → verification`

The agent supports picking a colored cube, stacking one cube on another, returning the arm home, and acknowledging stop commands locally. It includes a reproducible offline parser and can optionally connect to an OpenAI-compatible local LLM endpoint.

Model output never drives the robot directly: every plan must pass schema validation and perception checks before action dispatch.

> This synchronous teaching notebook prevents new actions after `stop`. A production deployment would also require asynchronous cancellation of an action already in progress.

#### Recommended Hardware

An AMD GPU supported by ROCm, such as an AMD Radeon™ GPU or AMD Ryzen™ AI processor with integrated Radeon graphics.

#### Software Environment

OS: Ubuntu 24.04 LTS  
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html). The Genesis Simulation image provides ROCm, PyTorch, and `genesis-world==1.3.1`.

## Goals

- Convert natural-language commands into constrained JSON plans.
- Reject invalid or unsafe plans before robot control.
- Confirm requested objects with ROCm visual perception.
- Gate every lift on a secure 8×8 fingertip tactile reading.
- Execute Genesis pick, stack, home, and stop behaviors.
- Verify visual, tactile, and motion outcomes in structured results.

In [ ]:
import os
import re
import json
import time
import uuid
import logging
import warnings
from pathlib import Path

os.environ.setdefault("TI_LOG_LEVEL", "error")
warnings.filterwarnings("ignore")

import numpy as np
import requests
import torch
import genesis as gs
import genesis.utils.geom as gu
from tqdm.auto import tqdm
from genesis.utils.misc import tensor_to_array

from helpers.physisim_hud import (
    FFmpegHUDWriter,
    GPUMonitor,
    build_vision_thumbnails,
    compose_hud_frame,
)
from helpers.physisim_widget import LiveAgentController

os.makedirs("Videos", exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PLANNER_MODE = os.getenv("PLANNER_MODE", "offline").lower()  # offline | llm
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "http://127.0.0.1:8081").rstrip("/")
LLM_MODEL = os.getenv("LLM_MODEL", "local-model")
LLM_TIMEOUT_SECONDS = float(os.getenv("LLM_TIMEOUT_SECONDS", "30"))

print("=== Runtime receipt ===")
print("PyTorch device :", DEVICE)
print("HIP            :", getattr(torch.version, "hip", None))
if DEVICE.type == "cuda":
    print("GPU            :", torch.cuda.get_device_name(0))
print("Planner mode   :", PLANNER_MODE)
print("LLM endpoint   :", LLM_BASE_URL)
print("LLM model      :", LLM_MODEL)

## 1. Build the Genesis capstone scene

The action layer reuses the Franka DOF groups, PD gains, IK orientation, and camera conventions from PhySim02–03. Three colored cubes provide deterministic targets for perception and language planning.

> Run this section once per kernel. Restart the kernel before rebuilding the scene.

In [ ]:
assert "scene" not in globals(), "Scene already exists. Restart the kernel before rebuilding it."

gs.init(backend=gs.amdgpu, theme="light", seed=0)
gs.logger._logger.setLevel(logging.WARNING)

CUBE_SIZE = 0.04
CUBE_HALF = CUBE_SIZE / 2.0
EE_QUAT = np.array([0.0, 1.0, 0.0, 0.0])
GRIPPER_OPEN = 0.04

cube_specs = {
    "red":   {"pos": (0.55, -0.15, CUBE_HALF), "color": (1.0, 0.0, 0.0, 1.0)},
    "green": {"pos": (0.55,  0.00, CUBE_HALF), "color": (0.0, 1.0, 0.0, 1.0)},
    "blue":  {"pos": (0.55,  0.15, CUBE_HALF), "color": (0.0, 0.0, 1.0, 1.0)},
}

scene = gs.Scene(
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(3, -1, 1.5),
        camera_lookat=(0.0, 0.0, 0.5),
        camera_fov=30,
        max_FPS=60,
    ),
    sim_options=gs.options.SimOptions(dt=0.01, substeps=4),
    rigid_options=gs.options.RigidOptions(
        box_box_detection=True,
        constraint_timeconst=0.01,
    ),
    show_viewer=False,
)

plane = scene.add_entity(gs.morphs.Plane())
cube_entities = {}
for name, spec in cube_specs.items():
    cube_entities[name] = scene.add_entity(
        gs.morphs.Box(size=(CUBE_SIZE,) * 3, pos=spec["pos"]),
        surface=gs.surfaces.Default(color=spec["color"]),
    )

franka = scene.add_entity(
    gs.morphs.MJCF(file="xml/franka_emika_panda/panda.xml"),
)
cam = scene.add_camera(
    res=(640, 480),
    pos=(3, -1, 1.5),
    lookat=(0, 0, 0.5),
    fov=30,
    GUI=True,
)

# Same tactile contract taught in PhySim05.
CONTACT_THRESH_M = 5e-4
CONTACT_SECURE_TAXELS = 12
GRIP_STIFFNESS_N_PER_M = 5000.0
GRIP_CLOSE_TIMEOUT = 150

probe_normal = (0.0, -1.0, 0.0)
probe_local_pos = gu.generate_grid_points_on_plane(
    lo=(-0.006, 0.0, 0.04),
    hi=(0.008, 0.0, 0.05),
    normal=probe_normal,
    nx=8,
    ny=8,
)
tracked_cube_links = tuple(int(entity.base_link_idx) for entity in cube_entities.values())
tactile_options = dict(
    probe_local_pos=probe_local_pos,
    probe_local_normal=probe_normal,
    probe_radius=0.002,
    track_link_idx=tracked_cube_links,
    n_sample_points=1000,
    lambda_d=5000.0,
    lambda_s=4000.0,
    dilate_scale=1.0,
    shear_scale=1.0,
    normal_exponent=1.0,
    compressibility=0.8,
    draw_debug=False,
)
left_tactile = scene.add_sensor(
    gs.sensors.ElastomerTaxel(
        entity_idx=franka.idx,
        link_idx_local=franka.get_link("left_finger").idx_local,
        **tactile_options,
    )
)
right_tactile = scene.add_sensor(
    gs.sensors.ElastomerTaxel(
        entity_idx=franka.idx,
        link_idx_local=franka.get_link("right_finger").idx_local,
        **tactile_options,
    )
)

scene.build()
print("Genesis scene built with cubes, Franka Panda, camera, and two 8×8 tactile pads")

## 2. Configure the Franka controller

The agent reuses the control model from PhySim02–03. The seven arm joints and two gripper joints are controlled separately, while shared PD gains stabilize the complete nine-DOF system.

We also save the initial joint configuration. The `home` behavior later plans a path back to this known pose instead of relying on a hard-coded posture.

In [ ]:
motors_dof = np.arange(7)
fingers_dof = np.arange(7, 9)
end_effector = franka.get_link("hand")

franka.set_dofs_kp(
    np.array([4500, 4500, 3500, 3500, 2000, 2000, 2000, 100, 100])
)
franka.set_dofs_kv(
    np.array([450, 450, 350, 350, 200, 200, 200, 10, 10])
)
franka.set_dofs_force_range(
    np.array([-87, -87, -87, -87, -12, -12, -12, -100, -100]),
    np.array([87, 87, 87, 87, 12, 12, 12, 100, 100]),
)


def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return tensor_to_array(value)
    return np.asarray(value)


INITIAL_QPOS = to_numpy(franka.get_qpos()).reshape(-1).astype(np.float64)
INITIAL_QPOS[-2:] = GRIPPER_OPEN
franka.set_qpos(INITIAL_QPOS)
for _ in range(20):
    scene.step()

print("Initial joint configuration:", INITIAL_QPOS.round(3).tolist())

## 3. Prepare the perception bridge

PhySim06 needs only the target-record contract from PhySim05, so the full dashboard is not repeated here. We render RGB plus entity segmentation, normalize the buffer layout, and use segmentation once to calibrate each cube's rendered color.

This calibration handles lighting and material differences between ideal colors such as `(1, 0, 0)` and the actual pixels seen by the camera.

In [ ]:
def squeeze_buffer(value):
    arr = to_numpy(value)
    if arr.ndim == 4 and arr.shape[0] == 1:
        arr = arr[0]
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = arr[..., 0]
    return arr


def as_rgb_uint8(value):
    arr = squeeze_buffer(value)[..., :3]
    if arr.dtype != np.uint8:
        arr = arr.astype(np.float32)
        if arr.size and float(np.nanmax(arr)) <= 1.5:
            arr = arr * 255.0
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    return arr


def render_perception_buffers():
    rgb, depth, segmentation, normal = cam.render(
        rgb=True,
        depth=True,
        segmentation=True,
        normal=True,
        colorize_seg=False,
    )
    return as_rgb_uint8(rgb), squeeze_buffer(segmentation).astype(np.int32)


initial_rgb, initial_seg = render_perception_buffers()
target_colors = {}
for name, entity in cube_entities.items():
    mask = initial_seg == int(entity.idx + 1)
    target_colors[name] = (
        (initial_rgb[mask].mean(axis=0) / 255.0).tolist()
        if mask.any()
        else list(cube_specs[name]["color"][:3])
    )

print("Calibrated target colors:", target_colors)

### 3.1 Convert visual evidence into target records

`color_lock()` produces image-space evidence: visibility, matching-pixel count, bounding box, and centroid. Only after an object is visually confirmed do we attach its exact Genesis world position.

As in PhySim05, `position_source="genesis_entity_state"` makes the simulator shortcut explicit. A physical robot would replace this field with calibrated depth or pose-estimation output.

In [ ]:
def color_lock(rgb, target_rgb, tolerance=0.30, min_pixels=8):
    # Genesis render buffers may use negative strides; PyTorch requires contiguous host memory.
    contiguous_rgb = np.ascontiguousarray(np.asarray(rgb)[..., :3])
    image = torch.from_numpy(contiguous_rgb).to(DEVICE, dtype=torch.float32) / 255.0
    target = torch.tensor(target_rgb, dtype=torch.float32, device=DEVICE)
    mask = torch.linalg.norm(image - target, dim=-1) < tolerance
    count = int(mask.sum().item())

    if count < min_pixels:
        return {"visible": False, "pixel_count": count, "bbox": None, "centroid_px": None}

    ys, xs = torch.where(mask)
    return {
        "visible": True,
        "pixel_count": count,
        "bbox": [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())],
        "centroid_px": [float(xs.float().mean()), float(ys.float().mean())],
    }


def entity_world_position(name):
    pos = to_numpy(cube_entities[name].get_pos(relative=False)).reshape(-1)[:3]
    return pos.astype(float).tolist()


def build_target_record(name, rgb=None):
    rgb = render_perception_buffers()[0] if rgb is None else rgb
    lock = color_lock(rgb, target_colors[name])
    return {
        "name": name,
        **lock,
        "world_position": entity_world_position(name) if lock["visible"] else None,
        "position_source": "genesis_entity_state" if lock["visible"] else None,
    }


for color in cube_entities:
    print(color, build_target_record(color, initial_rgb))

### 3.2 Reuse the tactile contract from PhySim05

The agent reads both 8×8 fingertip pads after each close-gripper step. `tactile_reduce()` returns the same five fields introduced in PhySim05: contact count, total taxels, force estimate, peak displacement, and `secure`.

Keeping this interface identical lets the action layer consume tactile evidence without knowing how the individual taxels are arranged.

In [ ]:
def read_tactile_displacement():
    left = to_numpy(left_tactile.read_ground_truth()).astype(np.float32)
    right = to_numpy(right_tactile.read_ground_truth()).astype(np.float32)
    return left, right


def tactile_reduce(
    left_disp,
    right_disp,
    contact_thresh_m=CONTACT_THRESH_M,
    secure_taxels=CONTACT_SECURE_TAXELS,
    grip_stiffness_N_per_m=GRIP_STIFFNESS_N_PER_M,
):
    left_host = np.ascontiguousarray(left_disp, dtype=np.float32)
    right_host = np.ascontiguousarray(right_disp, dtype=np.float32)
    left = torch.from_numpy(left_host).to(DEVICE).reshape(-1, 3)
    right = torch.from_numpy(right_host).to(DEVICE).reshape(-1, 3)
    left_magnitude = torch.linalg.norm(left, dim=-1)
    right_magnitude = torch.linalg.norm(right, dim=-1)

    n_contact = int(
        (left_magnitude > contact_thresh_m).sum().item()
        + (right_magnitude > contact_thresh_m).sum().item()
    )
    peak_m = torch.maximum(left_magnitude.max(), right_magnitude.max())
    grip_force_N = (left_magnitude.sum() + right_magnitude.sum()) * grip_stiffness_N_per_m
    return {
        "n_contact": n_contact,
        "n_taxels": int(left_magnitude.numel() + right_magnitude.numel()),
        "grip_force_N": float(grip_force_N.item()),
        "peak_mm": float(peak_m.item() * 1000.0),
        "secure": bool(n_contact >= secure_taxels),
    }


scene.step()
air_tactile = tactile_reduce(*read_tactile_displacement())
print("Open-air tactile receipt:", air_tactile)
assert air_tactile["n_taxels"] == 128
assert air_tactile["secure"] is False

## 4. Define a constrained plan language

Natural language is flexible, but robot handlers need predictable inputs. The planner therefore emits one of four small JSON shapes:

```json
{"intent": "pick", "object": "blue"}
{"intent": "stack", "object": "blue", "target": "red"}
{"intent": "home"}
{"intent": "stop"}
```

`validate_plan()` is the trust boundary between language and control. It rejects unknown keys, unsupported colors, missing fields, and attempts to stack an object on itself before any perception or motion code runs.

In [ ]:
ALLOWED_INTENTS = {"pick", "stack", "home", "stop"}
ALLOWED_KEYS = {"intent", "object", "target"}
COLORS = set(cube_entities)
COLOR_SYNONYMS = {
    "r": "red", "red": "red", "crimson": "red",
    "g": "green", "green": "green",
    "b": "blue", "blue": "blue",
}
STOP_RE = re.compile(r"\b(stop|halt|freeze|abort|emergency)\b", re.IGNORECASE)
NOISE_TEXT = {"", "uh", "um", "hmm", "hello", "hi", "thanks", "thank you", "okay", "ok"}


def normalize_text(text):
    return re.sub(r"[\s\.,!?]+$", "", (text or "").strip().lower()).strip()


def normalize_color(value):
    return COLOR_SYNONYMS.get(str(value).strip().lower()) if value is not None else None


def validate_plan(raw):
    if not isinstance(raw, dict):
        return None, ["plan must be a JSON object"]

    errors = []
    extra = set(raw) - ALLOWED_KEYS
    if extra:
        errors.append(f"unknown keys: {sorted(extra)}")

    intent = str(raw.get("intent", "")).strip().lower()
    if intent not in ALLOWED_INTENTS:
        errors.append(f"unsupported intent: {intent!r}")

    obj = normalize_color(raw.get("object")) if "object" in raw else None
    target = normalize_color(raw.get("target")) if "target" in raw else None

    if intent == "pick" and obj not in COLORS:
        errors.append(f"pick requires a supported object color, got {raw.get('object')!r}")
    if intent == "stack":
        if obj not in COLORS:
            errors.append(f"stack requires a supported object color, got {raw.get('object')!r}")
        if target not in COLORS:
            errors.append(f"stack requires a supported target color, got {raw.get('target')!r}")
        if obj is not None and obj == target:
            errors.append("stack object and target must differ")
    if intent in {"home", "stop"} and ("object" in raw or "target" in raw):
        errors.append(f"{intent} must not include object or target")

    if errors:
        return None, errors

    plan = {"intent": intent}
    if obj is not None:
        plan["object"] = obj
    if target is not None:
        plan["target"] = target
    return plan, []


for example in [
    {"intent": "pick", "object": "blue"},
    {"intent": "stack", "object": "red", "target": "red"},
    {"intent": "home", "object": "green"},
]:
    plan, errors = validate_plan(example)
    print(example, "->", plan or errors)

### 4.1 Start with a deterministic offline parser

The offline parser keeps the lab reproducible and makes intent routing easy to inspect. Regular expressions recognize the supported verbs, colors, and relations, then return the same dictionary shape expected from an LLM.

It is intentionally limited: unsupported language produces `{"intent": "unknown"}` and is rejected by the same validator used for model output.

In [ ]:
PICK_RE = re.compile(r"\b(?:pick|grab|get)\b.*?\b(red|green|blue|r|g|b)\b", re.I)
STACK_RE = re.compile(
    r"\b(?:stack|place|put)\b.*?\b(red|green|blue|r|g|b)\b.*?"
    r"\b(?:on|onto|above)\b.*?\b(red|green|blue|r|g|b)\b",
    re.I,
)
HOME_RE = re.compile(r"\b(?:go\s+home|home|rest)\b", re.I)


def offline_rule_parser(text):
    normalized = normalize_text(text)
    match = STACK_RE.search(normalized)
    if match:
        return {
            "intent": "stack",
            "object": normalize_color(match.group(1)),
            "target": normalize_color(match.group(2)),
        }

    match = PICK_RE.search(normalized)
    if match:
        return {"intent": "pick", "object": normalize_color(match.group(1))}

    if HOME_RE.search(normalized):
        return {"intent": "home"}
    return {"intent": "unknown"}


for command in ["pick green", "stack blue on red", "go home", "pick purple"]:
    print(command, "->", offline_rule_parser(command))

### 4.2 Optionally connect an OpenAI-compatible LLM

`llm_rule_parser()` sends the same task to a local `/v1/chat/completions` endpoint. A short system prompt limits the response vocabulary, but prompt instructions alone are not a safety boundary—returned JSON still passes through `validate_plan()`.

The lab does not download or start a model automatically. Configure `LLM_BASE_URL`, `LLM_MODEL`, and `PLANNER_MODE=llm` when a compatible local server is available.

In [ ]:
LLM_SYSTEM_PROMPT = """You route commands for a simulated Franka arm.
Return exactly one JSON object and no other text.
Allowed forms:
{"intent":"pick","object":"red|green|blue"}
{"intent":"stack","object":"red|green|blue","target":"red|green|blue"}
{"intent":"home"}
{"intent":"stop"}
The stack object and target must differ.
"""


def llm_rule_parser(text):
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": LLM_SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        "temperature": 0.0,
        "max_tokens": 64,
    }
    response = requests.post(
        f"{LLM_BASE_URL}/v1/chat/completions",
        json=payload,
        timeout=LLM_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    content = response.json()["choices"][0]["message"]["content"].strip()
    start, end = content.find("{"), content.rfind("}")
    if start < 0 or end < start:
        raise ValueError(f"model did not return a JSON object: {content!r}")
    return json.loads(content[start : end + 1])

### 4.3 Download the GGUF model on demand

The Docker image contains `llama-server`, but deliberately does not embed the 2.02 GB model. Use the button below once per workspace to download:

- Repository: `bartowski/Llama-3.2-3B-Instruct-GGUF`
- File: `Llama-3.2-3B-Instruct-Q4_K_M.gguf`
- Destination: `/opt/workspace/PhySim/models/`
- Expected SHA256: `6c1a2b41161032677be168d354123594c0e6e67d2b9227c84f296ad037c728ff`

The download is opt-in, so Offline mode and automated notebook execution remain network-independent. The `models/` directory is ignored by Git and Docker build context. When the course directory is mounted from the host, the model persists after the container stops.

Llama 3.2 weights are governed by the [Meta Llama 3.2 Community License](https://www.llama.com/llama3_2/license/). Review and accept the applicable terms before downloading or using the model. The quantized file is redistributed by the referenced Hugging Face repository; it is not covered by this course's MIT license.

You may also call `download_llm_model()` directly to see normal Python progress output.

In [ ]:
import hashlib
import importlib
import site
import subprocess
import sys

import ipywidgets as widgets
from IPython.display import display

try:
    from huggingface_hub import hf_hub_download
except ModuleNotFoundError:
    print("huggingface_hub is missing; installing it into the current Jupyter kernel…")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--user",
            "huggingface-hub>=0.34,<2",
        ]
    )
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.insert(0, user_site)
    importlib.invalidate_caches()
    from huggingface_hub import hf_hub_download

GGUF_REPO_ID = "bartowski/Llama-3.2-3B-Instruct-GGUF"
GGUF_REVISION = "c346bfc2029e79ba6d7edf026cf01fe44242db0d"
GGUF_FILENAME = "Llama-3.2-3B-Instruct-Q4_K_M.gguf"
GGUF_SHA256 = "6c1a2b41161032677be168d354123594c0e6e67d2b9227c84f296ad037c728ff"
GGUF_DIR = Path("/opt/workspace/PhySim/models")


def file_sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def download_llm_model():
    """Download and verify the pinned Q4_K_M GGUF, then update notebook globals."""
    global LLAMA_MODEL_PATH
    GGUF_DIR.mkdir(parents=True, exist_ok=True)
    destination = GGUF_DIR / GGUF_FILENAME

    if destination.is_file() and file_sha256(destination) == GGUF_SHA256:
        print("Model already present and checksum verified:", destination)
    else:
        if destination.exists():
            destination.unlink()
        print("Downloading approximately 2.02 GB from Hugging Face…")
        downloaded = Path(
            hf_hub_download(
                repo_id=GGUF_REPO_ID,
                filename=GGUF_FILENAME,
                revision=GGUF_REVISION,
                local_dir=GGUF_DIR,
            )
        )
        actual_sha256 = file_sha256(downloaded)
        if actual_sha256 != GGUF_SHA256:
            raise RuntimeError(
                f"GGUF checksum mismatch: expected {GGUF_SHA256}, got {actual_sha256}"
            )
        destination = downloaded
        print("Download and checksum verification complete:", destination)

    LLAMA_MODEL_PATH = destination
    os.environ["LLAMA_MODEL_PATH"] = str(destination)
    return destination


download_model_button = widgets.Button(
    description="Download Llama 3.2 3B GGUF (2.02 GB)",
    button_style="warning",
    icon="download",
)
download_model_output = widgets.Output()


def _download_model_clicked(_button):
    download_model_button.disabled = True
    try:
        with download_model_output:
            download_model_output.clear_output()
            download_llm_model()
    except Exception as error:
        with download_model_output:
            print(f"{type(error).__name__}: {error}")
    finally:
        download_model_button.disabled = False


download_model_button.on_click(_download_model_clicked)
display(widgets.VBox([download_model_button, download_model_output]))

### 4.4 Start the bundled LLM endpoint from a JupyterHub Terminal

The image contains a pinned Vulkan build of `llama-server` at:

```text
/opt/llama/bin/llama-server
```

After the download button reports a verified model, open **JupyterLab → File → New → Terminal** and run:

```bash
bash helpers/start_llama_server.sh
```

The helper automatically uses:

```text
/opt/llama/bin/llama-server
/opt/workspace/PhySim/models/Llama-3.2-3B-Instruct-Q4_K_M.gguf
```

Keep that Terminal open. A successful server reports that it is listening on `127.0.0.1:8081`. In a second Terminal, verify:

```bash
curl http://127.0.0.1:8081/health
```

Then rerun the health-check cell below and select **LLM endpoint** in the live agent interface. Press `Ctrl+C` in the server Terminal when finished.

Advanced users can override `LLAMA_SERVER_BIN`, `LLAMA_MODEL_PATH`, `LLAMA_HOST`, or `LLAMA_PORT` before running the helper. Because the Terminal and notebook kernel share one JupyterHub container, the default loopback address is correct.

In [ ]:
LLAMA_SERVER_BIN = Path(
    os.getenv("LLAMA_SERVER_BIN", "/opt/llama/bin/llama-server")
).expanduser()
LLAMA_MODEL_PATH = Path(
    os.getenv(
        "LLAMA_MODEL_PATH",
        "/opt/workspace/PhySim/models/Llama-3.2-3B-Instruct-Q4_K_M.gguf",
    )
).expanduser()


def llm_endpoint_status(base_url=LLM_BASE_URL):
    status = {
        "base_url": base_url,
        "server_binary": str(LLAMA_SERVER_BIN),
        "server_binary_ready": LLAMA_SERVER_BIN.is_file() and os.access(LLAMA_SERVER_BIN, os.X_OK),
        "model": str(LLAMA_MODEL_PATH),
        "model_ready": LLAMA_MODEL_PATH.is_file(),
        "endpoint_ready": False,
        "error": None,
    }
    try:
        response = requests.get(f"{base_url}/health", timeout=2)
        response.raise_for_status()
        status["endpoint_ready"] = True
        status["health"] = response.json() if response.content else {"status": "ok"}
    except (requests.RequestException, ValueError) as error:
        status["error"] = f"{type(error).__name__}: {error}"
    return status


llm_status = llm_endpoint_status()
print(json.dumps(llm_status, indent=2))
if not llm_status["endpoint_ready"]:
    if not llm_status["model_ready"]:
        print("\nDownload the GGUF with the button in the previous section first.")
    print("\nThen open a JupyterHub Terminal and run:")
    print("  bash helpers/start_llama_server.sh")
else:
    print("\nLLM endpoint is ready. Select 'LLM endpoint' in the live interface.")

### 4.5 Apply local safety gates before either planner

The order below is adapted from the VVLA pipeline and is part of the safety design:

1. normalize the transcript;
2. recognize `stop` locally without waiting for a model;
3. reject empty or noise-like text;
4. call the selected parser;
5. validate the returned plan.

Connection failures, malformed JSON, and schema errors produce a receipt with `plan=None`. They never fall back silently to a robot action.

In [ ]:
def plan_command(text, mode=PLANNER_MODE):
    started = time.perf_counter()
    normalized = normalize_text(text)

    if STOP_RE.search(normalized):
        return {
            "planner": "local-stop-fast-path",
            "latency_ms": 0.0,
            "plan": {"intent": "stop"},
            "errors": [],
        }
    if normalized in NOISE_TEXT or not any(char.isalpha() for char in normalized):
        return {
            "planner": "noise-gate",
            "latency_ms": 0.0,
            "plan": None,
            "errors": ["noise-like input ignored"],
        }

    if mode not in {"offline", "llm"}:
        raise ValueError(f"unsupported planner mode: {mode}")
    parser = offline_rule_parser if mode == "offline" else llm_rule_parser
    planner_name = "offline-rule-parser" if mode == "offline" else "openai-compatible-llm"

    try:
        raw = parser(normalized)
        plan, errors = validate_plan(raw)
        return {
            "planner": planner_name,
            "latency_ms": (time.perf_counter() - started) * 1000,
            "raw": raw,
            "plan": plan,
            "errors": errors,
        }
    except (requests.RequestException, KeyError, ValueError, json.JSONDecodeError) as error:
        return {
            "planner": planner_name,
            "latency_ms": (time.perf_counter() - started) * 1000,
            "plan": None,
            "errors": [f"{type(error).__name__}: {error}"],
        }


for command in [
    "pick the green cube",
    "stack the blue cube on the red cube",
    "go home",
    "stop immediately",
    "stack red on red",
    "pick purple",
]:
    receipt = plan_command(command, mode="offline")
    print(command, "->", receipt["plan"] or receipt["errors"])

assert plan_command("stop", mode="llm")["planner"] == "local-stop-fast-path"

## 5. Build reusable Genesis action primitives

The action layer turns validated plans into small, testable motion functions. It reuses the PhySim03 control pattern:

- `inverse_kinematics()` computes a Franka joint target;
- `plan_path()` handles longer collision-aware approaches;
- arm-only position control handles short vertical motions;
- force control closes the gripper;
- every behavior returns structured data or raises an error before later phases run.

Separating these primitives from language parsing makes the same motion code usable with the offline parser, an LLM, or future ROS 2 input.

In [ ]:
PATH_WAYPOINTS = 150
SETTLE_STEPS = 80
REACH_STEPS = 100
_recording = False
_hud_writer = None
_hud_monitor = None
_live_agent_widget = None
_hud_frame_index = 0
_seg_ids = initial_seg.astype(np.int32)
HUD_THUMBNAILS = {
    "gray": (initial_rgb[..., :3].astype(np.float32) @ np.array([0.299, 0.587, 0.114])).astype(np.uint8),
    "segmentation": np.stack(
        [(_seg_ids * 37 + 11) % 256, (_seg_ids * 79 + 43) % 256, (_seg_ids * 131 + 97) % 256],
        axis=-1,
    ).astype(np.uint8),
}
HUD_THUMBNAILS["segmentation"][_seg_ids == 0] = 0
HUD_CONTEXT = {
    "user_input": "",
    "planner_raw": None,
    "plan": None,
    "stages": [],
    "status": "Ready",
    "contact_threshold": CONTACT_THRESH_M,
    "secure_taxels": CONTACT_SECURE_TAXELS,
}


def _render_step(status=None, *, force=False, capture=True):
    """Render Genesis and optionally update MP4 and notebook live HUD sinks."""
    global _hud_frame_index, HUD_THUMBNAILS
    _hud_frame_index += 1
    hud_due = (
        (_hud_writer is not None or _live_agent_widget is not None)
        and (force or _hud_frame_index % 4 == 0)
    )
    if hud_due:
        rgb_frame, depth_frame, seg_frame, normal_frame = cam.render(
            rgb=True,
            depth=True,
            segmentation=True,
            normal=True,
            colorize_seg=False,
        )
        HUD_THUMBNAILS = build_vision_thumbnails(
            to_numpy(rgb_frame),
            to_numpy(depth_frame),
            to_numpy(seg_frame),
            to_numpy(normal_frame),
        )
    else:
        rgb_frame, *_ = cam.render(
            rgb=True,
            depth=False,
            segmentation=False,
            normal=False,
        )

    if hud_due:
        left_now, right_now = read_tactile_displacement()
        tactile_now = tactile_reduce(
            left_now,
            right_now,
            contact_thresh_m=HUD_CONTEXT["contact_threshold"],
            secure_taxels=HUD_CONTEXT["secure_taxels"],
        )
        if status:
            HUD_CONTEXT["status"] = status
        frame = compose_hud_frame(
            to_numpy(rgb_frame),
            title="PhySim06 · Language-Guided Agent",
            status=HUD_CONTEXT["status"],
            user_input=HUD_CONTEXT["user_input"],
            planner_raw=HUD_CONTEXT["planner_raw"],
            plan=HUD_CONTEXT["plan"],
            stages=HUD_CONTEXT["stages"],
            thumbnails=HUD_THUMBNAILS,
            tactile=tactile_now,
            left_tactile=left_now,
            right_tactile=right_now,
            gpu=_hud_monitor.snapshot() if _hud_monitor is not None else None,
        )
        if _hud_writer is not None:
            _hud_writer.write(frame)
        if _live_agent_widget is not None:
            _live_agent_widget.update(
                frame,
                HUD_CONTEXT["status"],
                capture=capture,
            )
    return rgb_frame


def simulate_steps(count, render=True, status="Settling simulation"):
    for _ in range(count):
        scene.step()
        if render:
            _render_step(status)


def solve_ik(position, init_qpos=None):
    """Solve IK on the branch nearest the current or explicitly supplied pose."""
    seed = (
        to_numpy(franka.get_qpos()).reshape(-1).astype(np.float64)
        if init_qpos is None
        else np.asarray(init_qpos, dtype=np.float64).reshape(-1)
    )
    qpos = franka.inverse_kinematics(
        link=end_effector,
        pos=np.asarray(position, dtype=np.float64),
        quat=EE_QUAT,
        init_qpos=seed,
    )
    qpos = to_numpy(qpos).reshape(-1).astype(np.float64)
    if qpos.size < 9 or not np.isfinite(qpos).all():
        raise RuntimeError(f"IK failed for position {np.asarray(position).tolist()}")
    return qpos


def execute_planned_path(qpos, description):
    path, valid = franka.plan_path(
        qpos_goal=np.asarray(qpos, dtype=np.float64),
        num_waypoints=PATH_WAYPOINTS,
        return_valid_mask=True,
    )
    is_valid = bool(to_numpy(valid).reshape(-1)[0])
    if not is_valid:
        raise RuntimeError(f"path planning failed: {description}")
    for waypoint in tqdm(path, desc=description, leave=False, ncols=90):
        franka.control_dofs_position(waypoint)
        scene.step()
        _render_step(description)
    simulate_steps(SETTLE_STEPS, status=f"Settling after {description}")


def move_arm_direct(position, description):
    qpos = solve_ik(position)
    franka.control_dofs_position(qpos[:-2], motors_dof)
    for _ in tqdm(range(REACH_STEPS), desc=description, leave=False, ncols=90):
        scene.step()
        _render_step(description)
    return qpos


def open_gripper():
    franka.control_dofs_position(
        np.array([GRIPPER_OPEN, GRIPPER_OPEN]),
        fingers_dof,
    )
    simulate_steps(REACH_STEPS, status="Opening gripper")


def close_gripper(
    arm_qpos,
    timeout_steps=GRIP_CLOSE_TIMEOUT,
    contact_threshold=CONTACT_THRESH_M,
    secure_taxels=CONTACT_SECURE_TAXELS,
):
    """Close until both tactile pads report a secure grasp or timeout."""
    arm_qpos = np.asarray(arm_qpos)
    tactile = tactile_reduce(
        *read_tactile_displacement(),
        contact_thresh_m=contact_threshold,
        secure_taxels=secure_taxels,
    )

    for step in range(timeout_steps):
        franka.control_dofs_position(arm_qpos[:-2], motors_dof)
        franka.control_dofs_position(np.array([-0.03, -0.03]), fingers_dof)
        scene.step()
        tactile = tactile_reduce(
            *read_tactile_displacement(),
            contact_thresh_m=contact_threshold,
            secure_taxels=secure_taxels,
        )
        _render_step(
            f"Closing gripper · K4 contact {tactile['n_contact']}/{tactile['n_taxels']}"
        )
        if tactile["secure"]:
            return {
                "success": True,
                "secure": True,
                "steps": step + 1,
                "tactile": tactile,
            }

    return {
        "success": False,
        "secure": False,
        "steps": timeout_steps,
        "reason": "grasp timeout without secure tactile contact",
        "tactile": tactile,
    }


print("Motion primitives ready; lift is tactile-gated")

### 5.1 Compose primitives into pick and stack behaviors

A pick uses three end-effector waypoints relative to the cube center:

1. **pre-grasp** approaches from above with an open gripper;
2. **grasp** lowers to the object and applies finger force;
3. **lift** raises the held object.

Stacking first performs a pick, then computes the destination cube center one cube-height above the target. These offsets are teaching parameters for the fixed cube and Franka geometry, not general-purpose grasp planning.

In [ ]:
def grasp_poses(object_center):
    center = np.asarray(object_center, dtype=np.float64)
    return {
        "pre_grasp": center + np.array([0.0, 0.0, 0.23]),
        "grasp": center + np.array([0.0, 0.0, 0.11]),
        "lift": center + np.array([0.0, 0.0, 0.26]),
    }


def pick_object(
    record,
    contact_threshold=CONTACT_THRESH_M,
    secure_taxels=CONTACT_SECURE_TAXELS,
):
    if not record["visible"] or record["world_position"] is None:
        raise RuntimeError(f"cannot pick invisible object: {record['name']}")
    poses = grasp_poses(record["world_position"])

    pre_qpos = solve_ik(poses["pre_grasp"])
    pre_qpos[-2:] = GRIPPER_OPEN
    execute_planned_path(pre_qpos, f"approach {record['name']}")

    grasp_qpos = move_arm_direct(poses["grasp"], f"lower to {record['name']}")
    grasp_receipt = close_gripper(
        grasp_qpos,
        contact_threshold=contact_threshold,
        secure_taxels=secure_taxels,
    )
    if not grasp_receipt["success"]:
        raise RuntimeError(
            "tactile grasp gate failed: "
            f"{grasp_receipt['reason']} "
            f"(contacts={grasp_receipt['tactile']['n_contact']})"
        )

    move_arm_direct(poses["lift"], f"lift {record['name']}")
    return {
        "success": True,
        "object": record["name"],
        "phases": ["approach", "tactile-secure grasp", "lift"],
        "grasp": grasp_receipt,
    }


def place_object(source_name, desired_center):
    poses = grasp_poses(desired_center)
    move_arm_direct(poses["pre_grasp"], f"carry {source_name}")
    move_arm_direct(poses["grasp"], f"lower {source_name}")
    open_gripper()
    move_arm_direct(poses["lift"], f"retreat from {source_name}")
    return {"success": True, "placed_center": np.asarray(desired_center).tolist()}


def stack_objects(
    source_record,
    target_record,
    contact_threshold=CONTACT_THRESH_M,
    secure_taxels=CONTACT_SECURE_TAXELS,
):
    pick_result = pick_object(
        source_record,
        contact_threshold=contact_threshold,
        secure_taxels=secure_taxels,
    )
    target = np.asarray(target_record["world_position"], dtype=np.float64)
    desired_center = target + np.array([0.0, 0.0, CUBE_SIZE])
    place_result = place_object(source_record["name"], desired_center)
    return {
        "success": True,
        "source": source_record["name"],
        "target": target_record["name"],
        "pick": pick_result,
        "place": place_result,
    }


def go_home():
    home = INITIAL_QPOS.copy()
    home[-2:] = GRIPPER_OPEN
    execute_planned_path(home, "return home")
    return {"success": True}

### 5.2 Guard the recording lifecycle

Genesis cameras allow only one active recording at a time. The `_recording` flag prevents nested starts and ensures `stop_recording()` is safe to call from a `finally` block even when an action raises an exception.

`start_recording()` now opens two synchronized outputs: the raw Genesis camera MP4 and a `_hud.mp4` stream composed by `helpers/physisim_hud.py`. Separate `start_hud_session()` and `stop_hud_session()` helpers support the optional typed-command demo without starting a second Genesis recorder.

In [ ]:
def start_hud_session(path="Videos/interactive_session.mp4", fps=25):
    global _hud_writer, _hud_monitor, _hud_frame_index
    if _hud_writer is not None:
        raise RuntimeError("HUD recording is already active")
    _hud_frame_index = 0
    if _hud_monitor is None:
        _hud_monitor = GPUMonitor().start()
    _hud_writer = FFmpegHUDWriter(path, fps=fps).open()
    return path


def stop_hud_session():
    global _hud_writer, _hud_monitor
    if _hud_writer is not None:
        _hud_writer.close()
        _hud_writer = None
    if _hud_monitor is not None:
        _hud_monitor.stop()
        _hud_monitor = None


def start_recording(path="Videos/video_06.mp4", fps=25):
    global _recording
    if _recording:
        raise RuntimeError("camera recording is already active")
    cam.start_recording(save_to_filename=path, fps=fps)
    hud_path = str(Path(path).with_name(Path(path).stem + "_hud.mp4"))
    start_hud_session(hud_path, fps=fps)
    _recording = True
    return path, hud_path


def stop_recording():
    global _recording
    if _recording:
        cam.stop_recording()
        _recording = False
    stop_hud_session()

## 6. Observe before acting and verify afterward

Before dispatch, `observe_targets()` renders one frame and creates records for every object named in the plan. Any invisible object blocks the command.

After motion, `verify_outcome()` checks simulator state rather than trusting the handler's return value. Pick requires measurable lift; stack requires both vertical separation and horizontal alignment; home checks joint error relative to the saved initial configuration.

In [ ]:
def observe_targets(plan):
    rgb, _ = render_perception_buffers()
    names = []
    if plan["intent"] in {"pick", "stack"}:
        names.append(plan["object"])
    if plan["intent"] == "stack":
        names.append(plan["target"])

    records = {name: build_target_record(name, rgb) for name in names}
    errors = [f"{name} is not visible" for name, record in records.items() if not record["visible"]]
    return records, errors


def extract_grasp_receipt(action_result):
    if not isinstance(action_result, dict):
        return None
    if isinstance(action_result.get("grasp"), dict):
        return action_result["grasp"]
    pick_result = action_result.get("pick")
    if isinstance(pick_result, dict):
        return pick_result.get("grasp")
    return None


def verify_outcome(plan, before_positions, action_result=None):
    intent = plan["intent"]
    if intent == "stop":
        return {"success": True, "checks": {"stop": "no action dispatched"}}
    if intent == "home":
        current = to_numpy(franka.get_qpos()).reshape(-1)
        error = float(np.max(np.abs(current[:7] - INITIAL_QPOS[:7])))
        return {"success": error < 0.15, "checks": {"max_joint_error": error}}

    source = plan["object"]
    source_before = np.asarray(before_positions[source])
    source_after = np.asarray(entity_world_position(source))
    moved = float(np.linalg.norm(source_after - source_before))
    checks = {"object_moved_m": moved}
    success = moved > 0.015

    grasp = extract_grasp_receipt(action_result)
    tactile = grasp.get("tactile", {}) if grasp else {}
    checks["tactile_secure"] = bool(grasp and grasp.get("secure"))
    checks["tactile_n_contact"] = int(tactile.get("n_contact", -1))
    checks["tactile_peak_mm"] = float(tactile.get("peak_mm", -1.0))
    success = success and checks["tactile_secure"]

    if intent == "pick":
        checks["object_lift_m"] = float(source_after[2] - source_before[2])
        success = success and checks["object_lift_m"] > 0.02

    if intent == "stack":
        target_after = np.asarray(entity_world_position(plan["target"]))
        xy_error = float(np.linalg.norm(source_after[:2] - target_after[:2]))
        height_difference = float(source_after[2] - target_after[2])
        checks.update({"stack_xy_error_m": xy_error, "stack_height_difference_m": height_difference})
        success = success and xy_error < 0.06 and height_difference > CUBE_SIZE * 0.7

    return {"success": bool(success), "checks": checks}


print("Observation and multimodal verification functions ready")

## 7. Register intent handlers

`CommandRegistry` decouples plan names from implementation functions. Each supported intent has one handler, and unregistered intents fail closed.

The `stop` handler intentionally issues no joint command. In this synchronous notebook it acknowledges the request and prevents a new behavior from starting; a production controller would also need asynchronous interruption for an action already in progress.

In [ ]:
class CommandRegistry:
    def __init__(self):
        self.handlers = {}

    def register(self, intent):
        def decorator(function):
            self.handlers[intent] = function
            return function
        return decorator

    def dispatch(self, plan, targets, tactile_config=None):
        if plan["intent"] not in self.handlers:
            raise RuntimeError(f"no handler registered for {plan['intent']}")
        return self.handlers[plan["intent"]](plan, targets, tactile_config or {})


registry = CommandRegistry()


@registry.register("pick")
def handle_pick(plan, targets, tactile_config):
    return pick_object(targets[plan["object"]], **tactile_config)


@registry.register("stack")
def handle_stack(plan, targets, tactile_config):
    return stack_objects(
        targets[plan["object"]],
        targets[plan["target"]],
        **tactile_config,
    )


@registry.register("home")
def handle_home(plan, targets, tactile_config):
    return go_home()


@registry.register("stop")
def handle_stop(plan, targets, tactile_config):
    return {"success": True, "reason": "stop acknowledged; no joint command issued"}


print("Registered intents:", sorted(registry.handlers))

## 8. Connect the complete guarded pipeline

`handle_command()` is the orchestrator. It records each stage in order:

`planner → perception → action → verification`

Every result carries a trace ID and latency. Early failures return immediately without adding an action stage. Action exceptions are converted into structured failure receipts, and recording is always closed in `finally`.

In [ ]:
def handle_command(
    text,
    mode=PLANNER_MODE,
    record=False,
    video_path="Videos/video_06.mp4",
    tactile_overrides=None,
):
    trace_id = uuid.uuid4().hex[:8]
    started = time.perf_counter()
    stages = []
    tactile_config = {
        "contact_threshold": CONTACT_THRESH_M,
        "secure_taxels": CONTACT_SECURE_TAXELS,
    }
    if tactile_overrides:
        tactile_config.update(tactile_overrides)
    HUD_CONTEXT.update(
        {
            "user_input": text,
            "planner_raw": None,
            "plan": None,
            "stages": [],
            "status": "Planning command",
            "contact_threshold": tactile_config["contact_threshold"],
            "secure_taxels": tactile_config["secure_taxels"],
        }
    )

    planner_receipt = plan_command(text, mode=mode)
    stages.append({"stage": "planner", **planner_receipt})
    plan = planner_receipt.get("plan")
    HUD_CONTEXT.update(
        {
            "planner_raw": planner_receipt.get("raw") or planner_receipt.get("errors"),
            "plan": plan,
            "stages": list(stages),
            "status": "Plan validated" if plan is not None else "Planner rejected command",
        }
    )
    if plan is None:
        return {
            "trace_id": trace_id,
            "success": False,
            "stages": stages,
            "total_latency_ms": (time.perf_counter() - started) * 1000,
        }

    targets, perception_errors = observe_targets(plan)
    stages.append({"stage": "perception", "targets": targets, "errors": perception_errors})
    HUD_CONTEXT.update(
        {
            "stages": list(stages),
            "status": "Targets confirmed" if not perception_errors else "Perception blocked action",
        }
    )
    if perception_errors:
        return {
            "trace_id": trace_id,
            "success": False,
            "plan": plan,
            "stages": stages,
            "total_latency_ms": (time.perf_counter() - started) * 1000,
        }

    before_positions = {
        name: record_data["world_position"]
        for name, record_data in targets.items()
        if record_data["world_position"] is not None
    }

    action_result = None
    HUD_CONTEXT["status"] = f"Executing {plan['intent']}"
    if record:
        start_recording(video_path)
    try:
        action_result = registry.dispatch(plan, targets, tactile_config=tactile_config)
    except Exception as error:
        action_result = {
            "success": False,
            "error": f"{type(error).__name__}: {error}",
        }
    finally:
        if record:
            stop_recording()
    stages.append({"stage": "action", "result": action_result})
    HUD_CONTEXT.update(
        {
            "stages": list(stages),
            "status": "Action completed" if action_result.get("success") else "Action failed",
        }
    )

    if action_result.get("success"):
        verification = verify_outcome(plan, before_positions, action_result=action_result)
    else:
        verification = {"success": False, "checks": {"skipped": "action failed"}}
    stages.append({"stage": "verification", **verification})

    return {
        "trace_id": trace_id,
        "success": bool(action_result.get("success") and verification["success"]),
        "plan": plan,
        "stages": stages,
        "total_latency_ms": (time.perf_counter() - started) * 1000,
    }


# These checks exercise planning and the local stop path without moving the robot.
for text in ["stop immediately", "unsupported request"]:
    result = handle_command(text, mode="offline", record=False)
    print(text, "->", result["success"], [stage["stage"] for stage in result["stages"]])

stop_result = handle_command("stop", mode="llm", record=False)
assert stop_result["plan"] == {"intent": "stop"}
assert stop_result["stages"][0]["planner"] == "local-stop-fast-path"

## 9. Run an end-to-end mission

The mission asks the agent to stack the blue cube on the red cube, records the manipulation, verifies the final geometry, and then returns the arm home.

This Run All mission explicitly uses the deterministic Offline planner so it remains reproducible even when an LLM endpoint is configured. Use the live language-agent interface later in the notebook to select and test the local LLM endpoint.

In [ ]:
from IPython.display import Video, display

video_path = "Videos/video_06.mp4"
hud_video_path = "Videos/video_06_hud.mp4"
mission = handle_command(
    "stack the blue cube on the red cube",
    mode="offline",  # deterministic Run All path; use the live widget to test the LLM endpoint
    record=True,
    video_path=video_path,
)

print("Mission summary:")
print(json.dumps({
    "trace_id": mission["trace_id"],
    "success": mission["success"],
    "plan": mission.get("plan"),
    "total_latency_ms": mission["total_latency_ms"],
}, indent=2))

for stage in mission["stages"]:
    if stage["stage"] in {"action", "verification"}:
        print(stage)

home_result = handle_command("go home", mode="offline", record=False)
print("Home result:", home_result["success"])

action_stage = next(stage for stage in mission["stages"] if stage["stage"] == "action")
verification_stage = next(stage for stage in mission["stages"] if stage["stage"] == "verification")
grasp_receipt = action_stage["result"]["pick"]["grasp"]

assert grasp_receipt["secure"] is True
assert grasp_receipt["tactile"]["n_contact"] >= CONTACT_SECURE_TAXELS
assert verification_stage["checks"]["tactile_secure"] is True
assert mission["success"], f"stack mission failed: {mission['stages']}"
assert home_result["success"], f"home behavior failed: {home_result['stages']}"

if os.path.exists(video_path):
    print("Raw Genesis mission")
    display(Video(video_path, embed=True, width=720))
else:
    print("No raw video was written because the mission stopped before action dispatch.")

if os.path.exists(hud_video_path):
    print("AI BRAIN mission HUD")
    display(Video(hud_video_path, embed=True, width=960))
else:
    print("No HUD video was written.")

## 10. Live language-agent interface

This widget sends commands through the same guarded pipeline used by the tested mission:

`text → planner → validate → visual confirmation → tactile-gated action → verification`

Use **Offline** for the self-contained parser or select **LLM endpoint** after configuring `LLM_BASE_URL`. The live HUD updates during robot motion and shows the command, validated plan, execution stage, GPU telemetry, vision thumbnails, and both tactile pads.

- **Run Command** executes the text field.
- **Home** and **Stop** use the same local safety paths as ordinary commands.
- **Reset Scene** restores all cubes and the initial robot pose.
- Tactile sliders change the actual close-gripper secure gate, not only the display.
- **Export HUD MP4** writes the captured interaction to `Videos/video_06_live_hud.mp4`.
- **Shutdown HUD** stops the GPU telemetry thread when the interaction is finished.

In [ ]:
from IPython.display import display

# Reset returns to the settled scene-start pose, before any mission moves the arm.
UPRIGHT_QPOS = INITIAL_QPOS.copy()
UPRIGHT_QPOS[-2:] = GRIPPER_OPEN

RANDOM_LAYOUT_X = (0.46, 0.62)
RANDOM_LAYOUT_Y = (-0.20, 0.20)
RANDOM_LAYOUT_MIN_DISTANCE = 0.09


def agent_cube_layout_positions(controller):
    """Return deterministic default or seeded-random cube positions."""
    if controller.scene_layout.value == "default":
        return {
            name: np.asarray(spec["pos"], dtype=np.float64).copy()
            for name, spec in cube_specs.items()
        }

    rng = np.random.default_rng(int(controller.layout_seed.value))
    positions = {}
    for name in cube_entities:
        for _ in range(200):
            candidate = np.array(
                [
                    rng.uniform(*RANDOM_LAYOUT_X),
                    rng.uniform(*RANDOM_LAYOUT_Y),
                    CUBE_HALF,
                ],
                dtype=np.float64,
            )
            if all(
                np.linalg.norm(candidate[:2] - other[:2]) >= RANDOM_LAYOUT_MIN_DISTANCE
                for other in positions.values()
            ):
                positions[name] = candidate
                break
        else:
            raise RuntimeError("Could not sample a collision-free seeded cube layout")
    return positions


def activate_agent_widget(controller):
    global _live_agent_widget, _hud_monitor, _hud_frame_index
    _live_agent_widget = controller
    _hud_frame_index = 0
    if _hud_monitor is None:
        _hud_monitor = GPUMonitor().start()


def agent_live_result_frame(controller, result):
    controller.last_result = result
    HUD_CONTEXT.update(
        {
            "stages": result.get("stages", []),
            "status": "Command succeeded" if result["success"] else "Command failed safely",
        }
    )
    for _ in range(40):
        scene.step()
        _render_step(HUD_CONTEXT["status"])


def agent_live_run(controller):
    activate_agent_widget(controller)
    result = handle_command(
        controller.command.value,
        mode=controller.planner_mode.value,
        record=False,
        tactile_overrides={
            "contact_threshold": controller.contact_threshold.value,
            "secure_taxels": controller.secure_taxels.value,
        },
    )
    agent_live_result_frame(controller, result)
    print(json.dumps({
        "trace_id": result["trace_id"],
        "success": result["success"],
        "plan": result.get("plan"),
    }, indent=2))


def agent_live_home(controller):
    controller.command.value = "home"
    agent_live_run(controller)


def agent_live_stop(controller):
    controller.command.value = "stop"
    agent_live_run(controller)


def agent_live_shutdown(controller):
    global _live_agent_widget, _hud_monitor
    if _hud_monitor is not None:
        _hud_monitor.stop()
        _hud_monitor = None
    _live_agent_widget = None
    controller.set_status("Live agent HUD stopped. Run, Home, Stop, or Reset will restart it.")


def agent_live_reset(controller):
    activate_agent_widget(controller)
    layout_positions = agent_cube_layout_positions(controller)
    for name, entity in cube_entities.items():
        entity.set_pos(layout_positions[name])
    franka.set_qpos(UPRIGHT_QPOS)
    franka.control_dofs_position(UPRIGHT_QPOS[:7], motors_dof)
    franka.control_dofs_position(
        np.array([GRIPPER_OPEN, GRIPPER_OPEN]),
        fingers_dof,
    )
    layout_status = (
        "Default layout"
        if controller.scene_layout.value == "default"
        else f"Random layout · seed {controller.layout_seed.value}"
    )
    HUD_CONTEXT.update(
        {
            "user_input": f"Reset Scene · {layout_status}",
            "planner_raw": None,
            "plan": {
                "intent": "reset",
                "layout": controller.scene_layout.value,
                "seed": int(controller.layout_seed.value),
            },
            "stages": [{"stage": "reset", "success": True}],
            "status": f"Scene reset · upright home · {layout_status}",
        }
    )
    for _ in range(50):
        scene.step()
        _render_step(HUD_CONTEXT["status"])

In [ ]:
agent_live = LiveAgentController(
    contact_threshold=CONTACT_THRESH_M,
    secure_taxels=CONTACT_SECURE_TAXELS,
    export_path="Videos/video_06_live_hud.mp4",
    export_fps=25,
)
agent_live.bind("run", agent_live_run)
agent_live.bind("home", agent_live_home)
agent_live.bind("stop", agent_live_stop)
agent_live.bind("reset", agent_live_reset)
agent_live.bind("shutdown", agent_live_shutdown)

activate_agent_widget(agent_live)
_render_step(
    "Live preview · enter a command or press Reset Scene.",
    force=True,
    capture=False,
)
display(agent_live.widget)

## 11. Optional terminal-style typed-command demo

The function below turns the notebook into a small REPL without changing the tested top-to-bottom path. It keeps one HUD recording open across multiple commands and saves `Videos/interactive_session.mp4` when you type `quit`.

Try `pick green`, `stack blue on red`, `home`, or `stop`. The function is defined but not called automatically, so `nbconvert` and **Run All** do not block on keyboard input.

```python
run_typed_demo(mode="offline")
# Or connect a local OpenAI-compatible endpoint first:
# run_typed_demo(mode="llm")
```

In [ ]:
def run_typed_demo(mode="offline", path="Videos/interactive_session.mp4"):
    """Run optional typed commands and record one continuous AI BRAIN HUD session."""
    command_count = 0
    start_hud_session(path, fps=25)
    try:
        while True:
            text = input("[YOU] > ").strip()
            if not text or text.lower() in {"quit", "exit", "q"}:
                break

            result = handle_command(text, mode=mode, record=False)
            command_count += 1
            print(
                json.dumps(
                    {
                        "trace_id": result["trace_id"],
                        "success": result["success"],
                        "plan": result.get("plan"),
                    },
                    indent=2,
                )
            )

            HUD_CONTEXT.update(
                {
                    "stages": result.get("stages", []),
                    "status": "Command succeeded" if result["success"] else "Command failed safely",
                }
            )
            for _ in range(40):
                scene.step()
                _render_step(HUD_CONTEXT["status"])
    finally:
        stop_hud_session()

    print(f"Saved {command_count} typed commands to {path}")
    if os.path.exists(path):
        display(Video(path, embed=True, width=960))
    return path

## 12. Test failure paths

A safe agent must demonstrate what it refuses to do. The following cases check unsupported colors, self-stacking, unknown intents, extra fields, and unsupported language.

Each invalid request must fail before action dispatch. The final assertion verifies this by confirming that the stage trace contains no `action` entry.

In [ ]:
# Failure-path exercises: all are blocked before robot action dispatch.
invalid_cases = [
    {"intent": "pick", "object": "purple"},
    {"intent": "stack", "object": "red", "target": "red"},
    {"intent": "dance"},
    {"intent": "home", "object": "blue"},
]

for raw in invalid_cases:
    plan, errors = validate_plan(raw)
    assert plan is None
    print("Blocked:", raw, "->", errors)

unsupported = handle_command("please fly away", mode="offline", record=False)
assert unsupported["success"] is False
assert all(stage["stage"] != "action" for stage in unsupported["stages"])
print("PASS: unsupported language produced no action stage")

### 12.1 Verify tactile timeout behavior

A syntactically valid command is still unsafe when the gripper cannot confirm contact. With the arm at home, closing in open air must reach the timeout with `secure=False`.

This demonstrates the difference between plan validation and execution-time safety: both are required before an agent can report success.

In [ ]:
franka.set_qpos(INITIAL_QPOS)
franka.control_dofs_position(np.array([GRIPPER_OPEN, GRIPPER_OPEN]), fingers_dof)
simulate_steps(20, render=False)
home_qpos = to_numpy(franka.get_qpos()).reshape(-1)
air_grasp = close_gripper(home_qpos, timeout_steps=30)
print("Open-air grasp receipt:", air_grasp)

assert air_grasp["success"] is False
assert air_grasp["secure"] is False
assert "timeout" in air_grasp["reason"]

open_gripper()

## Conclusions

You built a simulation-native physical AI agent with local language safety gates, ROCm visual perception, two 8×8 fingertip tactile pads, tactile-gated grasping, Genesis IK and manipulation behaviors, and structured multimodal verification. A lift occurs only after the taxel reduction confirms secure contact; open-air closure fails with a timeout receipt. The AI BRAIN HUD synchronizes scene motion with plans, stage receipts, GPU telemetry, vision thumbnails, and live tactile heatmaps. The notebook-native agent widget lets you submit commands, switch planner modes, reset/home/stop the scene, tune the real tactile gate, and export captured frames. A terminal-style typed-command function remains available as an optional alternative without blocking automated notebook execution.

## Acknowledgements

This notebook adapts material and design patterns from two original projects:

- The ROCm perception and Genesis agent workshop in [`AI_LABS/vision_kernels_rocm/ROCm_Physical_AI_Agent_Workshop.ipynb`](https://gitenterprise.xilinx.com/ssw-mktg/igpu-training-env), from the [ssw-mktg/igpu-training-env](https://gitenterprise.xilinx.com/ssw-mktg/igpu-training-env) repository.
- The intent validation, local stop path, command dispatch, and behavior architecture from the [ahaidous/aai-vvla-pipeline](https://gitenterprise.xilinx.com/ahaidous/aai-vvla-pipeline) repository.

We thank the contributors to both repositories for the original Physical AI workshop and VVLA pipeline material used as the foundation for this Genesis 1.3.1 simulation adaptation.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.  
SPDX-License-Identifier: MIT